<h1 style="text-align:center;">Kuhn Poker : extensive-form objects before CFR</h1>

*Sources* :
- [Zinkevich et al. (2007), Regret Minimization in Games with Incomplete Information](https://poker.cs.ualberta.ca/publications/NIPS07-cfr.pdf)
- [Kuhn (1950), A simplified two-person poker](https://www.jstor.org/stable/2236819)
- [Neller & Lanctot (2013), An Introduction to Counterfactual Regret Minimization](http://modelai.gettysburg.edu/2013/cfr/cfr.pdf)

*Context* : construct the finite extensive-form game

$$
\Gamma=(N,\Omega,\rho,H,Z,P,A,\tau,u,\mathcal{I})
$$

## Sumary :

* [**1. Objects of the game**](#1)
    * [1.1. Players and cards](#1_1)
    * [1.2. Chance space](#1_2)
    * [1.3. Histories](#1_3)
    * [1.4. Action correspondence](#1_4)
    * [1.5. Transition map](#1_5)
    * [1.6. Player function](#1_6)
    * [1.7. Utility function](#1_7)
    * [1.8. Information sets](#1_8)
    * [1.9. State object](#1_9)
* [**2. Validation**](#2)
    * [2.1. Local checks](#2_1)
    * [2.2. Enumerating histories](#2_2)
    * [2.3. Enumerating information sets](#2_3)
* [**3. Appendix: CFR notation**](#3)

In [19]:
# Modules
from itertools import permutations
from dataclasses import dataclass
from typing import Optional
import numpy as np  

<a id='1'></a>
# 1. Objects of the game

We define each object of:

$$
\Gamma=(N,\Omega,\rho,H,Z,P,A,\tau,u,\mathcal{I})
$$

<a id='1_1'></a>
### 1.1. Players and cards

- Player set: $\qquad N=\{0,1\}$

- Card set : $ \qquad \mathcal{C}=\{J,Q,K\}$

- Rank map: $ \qquad r:\mathcal{C}\to\{0,1,2\}, \qquad r(J)=0,\quad r(Q)=1,\quad r(K)=2$

In [20]:
PLAYERS = [0, 1]

CARDS = ["J", "Q", "K"]
CARD_RANK = {"J": 0, "Q": 1, "K": 2}

assert PLAYERS == [0, 1]
assert CARD_RANK["J"] < CARD_RANK["Q"] < CARD_RANK["K"]

<a id='1_2'></a>
### 1.2. Chance space

- Chance outcome is an ordered deal: $\qquad \omega=(c_0,c_1)$

- Chance space is: $\qquad \Omega=\{(c_0,c_1)\in\mathcal{C}^2:c_0\neq c_1\}$

Since $\quad |\mathcal{C}|=3$: $|\Omega|=3\times 2=6$ :

- Chance distribution is uniform: $\qquad \rho(\omega)=\frac{1}{6},\qquad \omega\in\Omega$

In [21]:
def all_deals() -> list[tuple[str, str]]:
    '''Return Omega, the ordered private-card deals.'''
    return list(permutations(CARDS, 2))


OMEGA = all_deals()
CHANCE_PROB = {omega: 1 / len(OMEGA) for omega in OMEGA}

In [22]:
print(len(OMEGA))
print(CHANCE_PROB)
print(np.sum([CHANCE_PROB[omega] for omega in OMEGA]))

6
{('J', 'Q'): 0.16666666666666666, ('J', 'K'): 0.16666666666666666, ('Q', 'J'): 0.16666666666666666, ('Q', 'K'): 0.16666666666666666, ('K', 'J'): 0.16666666666666666, ('K', 'Q'): 0.16666666666666666}
0.9999999999999999


<a id='1_3'></a>
### 1.3. Histories

- Action alphabet: $\qquad \mathcal{A}=\{c,b,f\}$

where $c$ denotes check/call, $b$ denotes bet, and $f$ denotes fold.

- Non-terminal histories: $\qquad H\setminus Z=\{\emptyset,c,b,cb\}$

- Terminal histories: $\qquad Z=\{cc,bc,bf,cbc,cbf\}$

- Full public-history set: $\qquad H=(H\setminus Z)\cup Z$

In [23]:
CHECK_CALL = "c"
BET = "b"
FOLD = "f"

ACTIONS = [CHECK_CALL, BET, FOLD]

NON_TERMINAL_HISTORIES = {"", "c", "b", "cb"}
TERMINAL_HISTORIES = {"cc", "bc", "bf", "cbc", "cbf"}
HISTORIES = NON_TERMINAL_HISTORIES | TERMINAL_HISTORIES


def is_terminal(history: str) -> bool:
    '''Indicator of z in Z.'''
    return history in TERMINAL_HISTORIES

<a id='1_4'></a>
### 1.4. Action correspondence

The legal-action correspondence is:

$$
A(h)=
\begin{cases}
\{c,b\}, & h\in\{\emptyset,c\},\\
\{c,f\}, & h\in\{b,cb\},\\
\emptyset, & h\in Z.
\end{cases}
$$

Thus $A:H\to 2^{\mathcal{A}}$.

In [24]:
def legal_actions(history: str) -> list[str]:
    '''Return A(h).'''
    if history in ("", "c"):
        return [CHECK_CALL, BET]
    if history in ("b", "cb"):
        return [CHECK_CALL, FOLD]
    if is_terminal(history):
        return []
    raise ValueError(f"Unknown history: {history!r}")

<a id='1_5'></a>
### 1.5. Transition map

For non-terminal $h$ and legal action $a\in A(h)$:

$$
\tau(h,a)=ha.
$$

The deterministic edges are:

$$
\emptyset\xrightarrow{c}c,\quad
\emptyset\xrightarrow{b}b,\quad
c\xrightarrow{c}cc,\quad
c\xrightarrow{b}cb,
$$

$$
b\xrightarrow{c}bc,\quad
b\xrightarrow{f}bf,\quad
cb\xrightarrow{c}cbc,\quad
cb\xrightarrow{f}cbf.
$$

In [25]:
def next_history(history: str, action: str) -> str:
    '''Return tau(h,a)=ha after checking a in A(h).'''
    if action not in legal_actions(history):
        raise ValueError(f"Illegal action {action!r} at history {history!r}")
    return history + action

<a id='1_6'></a>
### 1.6. Player function

The player-to-act function is:

$$
P:H\setminus Z\to N
$$

with:

$$
P(h)=
\begin{cases}
0, & h\in\{\emptyset,cb\},\\
1, & h\in\{c,b\}.
\end{cases}
$$

For $z\in Z$, no player acts.

In [26]:
def current_player(history: str) -> Optional[int]:
    '''Return P(h), or None for h in Z.'''
    if is_terminal(history):
        return None
    if history in ("", "cb"):
        return 0
    if history in ("c", "b"):
        return 1
    raise ValueError(f"Unknown history: {history!r}")

<a id='1_7'></a>
### 1.7. Utility function

Let $\omega=(c_0,c_1)\in\Omega$.

Define:

$$
s_0(\omega)=
\begin{cases}
1, & r(c_0)>r(c_1),\\
-1, & r(c_0)<r(c_1).
\end{cases}
$$

Player $0$'s terminal utility is:

$$
u_0(\omega,z)=
\begin{cases}
s_0(\omega), & z=cc,\\
2s_0(\omega), & z\in\{bc,cbc\},\\
1, & z=bf,\\
-1, & z=cbf.
\end{cases}
$$

The game is zero-sum:

$$
u_1(\omega,z)=-u_0(\omega,z).
$$

In [ ]:
def showdown_sign_player0(cards: tuple[str, str]) -> int:
    '''Return s_0(omega).'''
    c0, c1 = cards
    return 1 if CARD_RANK[c0] > CARD_RANK[c1] else -1


def payoff_player0(cards: tuple[str, str], history: str) -> int:
    '''Return u_0(omega,z).'''
    if not is_terminal(history):
        raise ValueError(f"Payoff is defined only for z in Z, got {history!r}")
    if history == "cc":
        return showdown_sign_player0(cards)
    if history in ("bc", "cbc"):
        return 2 * showdown_sign_player0(cards)
    if history == "bf":
        return 1
    if history == "cbf":
        return -1


def payoff(cards: tuple[str, str], history: str, player: int = 0) -> int:
    '''Return u_i(omega,z).'''
    u0 = payoff_player0(cards, history)
    return u0 if player == 0 else -u0

<a id='1_8'></a>
### 1.8. Information sets

For player $i$, two decision states $(\omega,h)$ and $(\omega',h')$ are indistinguishable if:

$$
(\omega,h)\sim_i(\omega',h')
\Longleftrightarrow
h=h'
\quad\text{and}\quad
\omega_i=\omega'_i.
$$

The information set containing $(\omega,h)$ is:

$$
I_i(\omega_i,h)
=
\{(\omega',h):\omega'_i=\omega_i,\ P(h)=i\}.
$$

We encode it by:

$$
\texttt{P\{i\}|\{card\}|\{history\}}.
$$

In [28]:
def infoset_key(cards: tuple[str, str], history: str) -> Optional[str]:
    '''Return the key representing I_i(card_i,h).'''
    player = current_player(history)
    if player is None:
        return None
    private_card = cards[player]
    return f"P{player}|{private_card}|{history}"

<a id='1_9'></a>
### 1.9. State object

A state is:

$$
x=(\omega,h)\in\Omega\times H.
$$

It bundles the previously defined maps:

$$
P(h),\quad A(h),\quad \tau(h,a),\quad u_i(\omega,h),\quad I_i(\omega_i,h).
$$

In [29]:
@dataclass(frozen=True)
class State:
    cards: tuple[str, str]
    history: str = ""

    @property
    def player(self) -> Optional[int]:
        return current_player(self.history)

    @property
    def terminal(self) -> bool:
        return is_terminal(self.history)

    def actions(self) -> list[str]:
        return legal_actions(self.history)

    def child(self, action: str) -> "State":
        return State(cards=self.cards, history=next_history(self.history, action))

    def infoset_key(self) -> Optional[str]:
        return infoset_key(self.cards, self.history)

    def utility(self, player: int = 0) -> int:
        return payoff(self.cards, self.history, player=player)


s = State(cards=("K", "J"))

<a id='2'></a>
# 2. Validation

We now verify the global structure induced by the local definitions.

<a id='2_1'></a>
### 2.1. Local checks

In [30]:
for h in NON_TERMINAL_HISTORIES:
    assert current_player(h) in PLAYERS
    assert len(legal_actions(h)) > 0

for z in TERMINAL_HISTORIES:
    assert current_player(z) is None
    assert legal_actions(z) == []

for omega in OMEGA:
    for z in TERMINAL_HISTORIES:
        assert payoff(omega, z, player=0) == -payoff(omega, z, player=1)

print("Local checks passed.")

Local checks passed.


<a id='2_2'></a>
### 2.2. Enumerating histories

Starting from $\emptyset$, recursively apply:

$$
h'=\tau(h,a),\qquad a\in A(h).
$$

In [31]:
def enumerate_histories(history: str = "") -> list[str]:
    '''Enumerate all histories reachable from history.'''
    histories = [history]
    if is_terminal(history):
        return histories
    for action in legal_actions(history):
        histories.extend(enumerate_histories(next_history(history, action)))
    return histories


histories = enumerate_histories()
assert set(histories) == HISTORIES

history_table = [
    {
        "history": h,
        "terminal": is_terminal(h),
        "player_to_act": current_player(h),
        "legal_actions": legal_actions(h),
    }
    for h in histories
]

history_table

[{'history': '',
  'terminal': False,
  'player_to_act': 0,
  'legal_actions': ['c', 'b']},
 {'history': 'c',
  'terminal': False,
  'player_to_act': 1,
  'legal_actions': ['c', 'b']},
 {'history': 'cc',
  'terminal': True,
  'player_to_act': None,
  'legal_actions': []},
 {'history': 'cb',
  'terminal': False,
  'player_to_act': 0,
  'legal_actions': ['c', 'f']},
 {'history': 'cbc',
  'terminal': True,
  'player_to_act': None,
  'legal_actions': []},
 {'history': 'cbf',
  'terminal': True,
  'player_to_act': None,
  'legal_actions': []},
 {'history': 'b',
  'terminal': False,
  'player_to_act': 1,
  'legal_actions': ['c', 'f']},
 {'history': 'bc',
  'terminal': True,
  'player_to_act': None,
  'legal_actions': []},
 {'history': 'bf',
  'terminal': True,
  'player_to_act': None,
  'legal_actions': []}]

<a id='2_3'></a>
### 2.3. Enumerating information sets

For every decision state $(\omega,h)$ with $h\notin Z$, group by:

$$
(P(h),\omega_{P(h)},h).
$$

In [32]:
def decision_states() -> list[State]:
    '''Return all states (omega,h) with h not in Z.'''
    return [
        State(cards=omega, history=h)
        for omega in OMEGA
        for h in histories
        if not is_terminal(h)
    ]


infoset_rows = []
for state in decision_states():
    infoset_rows.append({
        "cards": state.cards,
        "history": state.history,
        "player": state.player,
        "infoset": state.infoset_key(),
        "actions": state.actions(),
    })

infoset_rows = sorted(infoset_rows, key=lambda row: (row["player"], row["infoset"], row["cards"]))

summary = {}
for row in infoset_rows:
    key = (row["player"], row["infoset"])
    if key not in summary:
        summary[key] = {
            "player": row["player"],
            "infoset": row["infoset"],
            "number_of_states": 0,
            "possible_deals": [],
            "actions": row["actions"],
        }
    summary[key]["number_of_states"] += 1
    summary[key]["possible_deals"].append(row["cards"])

infoset_summary = list(summary.values())

assert len(infoset_summary) == 12
infoset_summary

[{'player': 0,
  'infoset': 'P0|J|',
  'number_of_states': 2,
  'possible_deals': [('J', 'K'), ('J', 'Q')],
  'actions': ['c', 'b']},
 {'player': 0,
  'infoset': 'P0|J|cb',
  'number_of_states': 2,
  'possible_deals': [('J', 'K'), ('J', 'Q')],
  'actions': ['c', 'f']},
 {'player': 0,
  'infoset': 'P0|K|',
  'number_of_states': 2,
  'possible_deals': [('K', 'J'), ('K', 'Q')],
  'actions': ['c', 'b']},
 {'player': 0,
  'infoset': 'P0|K|cb',
  'number_of_states': 2,
  'possible_deals': [('K', 'J'), ('K', 'Q')],
  'actions': ['c', 'f']},
 {'player': 0,
  'infoset': 'P0|Q|',
  'number_of_states': 2,
  'possible_deals': [('Q', 'J'), ('Q', 'K')],
  'actions': ['c', 'b']},
 {'player': 0,
  'infoset': 'P0|Q|cb',
  'number_of_states': 2,
  'possible_deals': [('Q', 'J'), ('Q', 'K')],
  'actions': ['c', 'f']},
 {'player': 1,
  'infoset': 'P1|J|b',
  'number_of_states': 2,
  'possible_deals': [('K', 'J'), ('Q', 'J')],
  'actions': ['c', 'f']},
 {'player': 1,
  'infoset': 'P1|J|c',
  'number_of_stat

<a id='3'></a>
# 3. Appendix: CFR notation

A behavioral strategy is:

$$
\sigma_i:\mathcal{I}_i\to\Delta(A(I)).
$$

For every $I\in\mathcal{I}_i$:

$$
\sum_{a\in A(I)}\sigma_i(I,a)=1,
\qquad
\sigma_i(I,a)\ge 0.
$$

Reach probability factorization:

$$
\pi^\sigma(h)=\pi_c(h)\pi_0^\sigma(h)\pi_1^\sigma(h).
$$

Counterfactual reach probability:

$$
\pi_{-i}^{\sigma}(h)
=
\pi_c(h)\prod_{j\neq i}\pi_j^\sigma(h).
$$

Counterfactual value:

$$
v_i(\sigma,I)
=
\sum_{h\in I}
\pi_{-i}^{\sigma}(h)
\sum_{z\in Z:h\sqsubset z}
\pi^\sigma(h,z)u_i(z).
$$

Instantaneous regret:

$$
r_t(I,a)=v_i(\sigma^t_{I\to a},I)-v_i(\sigma^t,I).
$$

Cumulative regret:

$$
R_T(I,a)=\sum_{t=1}^T r_t(I,a).
$$

Regret matching:

$$
\sigma_{T+1}(I,a)
=
\begin{cases}
\dfrac{R_T^+(I,a)}
{\sum_{a'\in A(I)}R_T^+(I,a')},
& \text{if }\sum_{a'}R_T^+(I,a')>0,\\
\dfrac{1}{|A(I)|},
& \text{otherwise}.
\end{cases}
$$